# Init

In [2]:
import pandas as pd
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"     
TP_SIZE = 2  # 你的GPU数量    
# Qwen/Qwen2.5-VL-7B-Instruct
# meta-llama/Llama-3.1-8B-Instruct
# deepseek-ai/DeepSeek-R1-Distill-Qwen-32B

print(f" 正在加载 Tokenizer 和模型 {MODEL_PATH} ... ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 初始化 vLLM 引擎
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TP_SIZE,
    dtype="bfloat16",
    max_model_len=1024,            # 上下文截断
    gpu_memory_utilization=0.85,  # 显存预占比例
    enforce_eager=True            # 关闭 CUDA Graph 防止 OOM
)

 正在加载 Tokenizer 和模型 meta-llama/Llama-3.1-8B-Instruct ... 
INFO 05-14 16:52:49 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 1024, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
INFO 05-14 16:52:50 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 05-14 16:52:50 [model.py:1678] Using max model len 1024
INFO 05-14 16:52:50 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-14 16:52:51 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-14 16:52:51 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-14 16:52:51 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-14 16:52:55 [vllm.py:1025] Cudagra

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(Worker_TP1 pid=1374427) INFO 05-14 16:53:09 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(Worker_TP1 pid=1374427) INFO 05-14 16:53:09 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.71s
(Worker_TP0 pid=1374426) INFO 05-14 16:53:10 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/2)
(Worker_TP0 pid=1374426) INFO 05-14 16:53:10 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(Worker_TP0 pid=1374426) INFO 05-14 16:53:10 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.71s
(Worker_TP0 pid=1374426) INFO 05-14 16:53:11 [default_loader.py:384] Loading weights took 1.65 seconds
(Worker_TP0 pid=1374426) INFO 05-14 16:53:11 [gpu_model_runner.py:4820] Model loading took 7.51 GiB memory and 3.349982 seconds
(Worker_TP0 pid=1374426) INFO 05-14 16:53:33 [gpu_worker.py:436] Available KV cache memory: 29.62 GiB
(EngineCore pid=1374408) INFO 05-14 16:53:33 [kv_cache_utils.py:1319] GPU KV cache siz

In [4]:
test_cases = [
    {"Sub": "car", "Rel": "on", "Obj": "street", "Expected": "Yes"},

    {"Sub": "person", "Rel": "wearing", "Obj": "hat", "Expected": "Yes"},
    {"Sub": "boat", "Rel": "flying in", "Obj": "room", "Expected": "No"},
    {"Sub": "cat", "Rel": "sitting on", "Obj": "table", "Expected": "Yes"},
    {"Sub": "pizza", "Rel": "eating", "Obj": "man", "Expected": "No"},
    {"Sub": "train", "Rel": "parked on", "Obj": "track", "Expected": "Yes"},
    {"Sub": "tree", "Rel": "riding", "Obj": "bike", "Expected": "No"},
    {"Sub": "dog", "Rel": "holding", "Obj": "branch", "Expected": "Yes"},
    {"Sub": "flower", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},

    {"Sub": "logo", "Rel": "of", "Obj": "banana", "Expected": "Yes"},

    {"Sub": "person", "Rel": "along", "Obj": "towel", "Expected": "No"},
    {"Sub": "vase", "Rel": "sitting on", "Obj": "airplane", "Expected": "No"},
    {"Sub": "bag", "Rel": "across", "Obj": "street", "Expected": "No"},
    {"Sub": "leg", "Rel": "of", "Obj": "branch", "Expected": "No"},
    {"Sub": "laptop", "Rel": "of", "Obj": "screen", "Expected": "No"},
    {"Sub": "girl", "Rel": "attached to", "Obj": "book", "Expected": "No"},
    {"Sub": "man", "Rel": "eating", "Obj": "surfboard", "Expected": "No"},
    {"Sub": "man", "Rel": "says", "Obj": "snowboard", "Expected": "No"},


    {"Sub": "animal", "Rel": "above", "Obj": "ear", "Expected": "No"},           # 讨论过的空间几何错误
    {"Sub": "horse", "Rel": "riding", "Obj": "boy", "Expected": "No"},           # 主客体颠倒：人骑马，马不骑人
    {"Sub": "food", "Rel": "eating", "Obj": "woman", "Expected": "No"},          # 主客体颠倒：食物不能吃人
    {"Sub": "shoe", "Rel": "walking on", "Obj": "sidewalk", "Expected": "No"},   # 拟人化陷阱：鞋子本身不会走
    {"Sub": "elephant", "Rel": "sitting on", "Obj": "chair", "Expected": "No"},  # 物理尺度陷阱：普通椅子承受不了大象
    {"Sub": "cup", "Rel": "holding", "Obj": "elephant", "Expected": "No"},       # 物理尺度陷阱：杯子装不下大象
    {"Sub": "motorcycle", "Rel": "wearing", "Obj": "helmet", "Expected": "No"},  # 拟人化陷阱：机器不穿戴衣物
    {"Sub": "wing", "Rel": "part of", "Obj": "dog", "Expected": "No"},           # 解剖学错误：狗没有翅膀
    {"Sub": "leg", "Rel": "attached to", "Obj": "wheel", "Expected": "No"},      # 生物与机械错乱
    {"Sub": "tree", "Rel": "growing on", "Obj": "dog", "Expected": "No"},        # 生物常识错误
    {"Sub": "boat", "Rel": "parked on", "Obj": "tree", "Expected": "No"},        # 极端边缘场景
    {"Sub": "building", "Rel": "wearing", "Obj": "hat", "Expected": "No"},       # 隐喻/卡通陷阱
    {"Sub": "car", "Rel": "in", "Obj": "box", "Expected": "No"},                 # 尺度陷阱：真车装不进普通纸箱
    {"Sub": "logo", "Rel": "painted on", "Obj": "horse", "Expected": "No"},      # 讨论过的边缘特例


    {"Sub": "bird", "Rel": "standing on", "Obj": "cow", "Expected": "Yes"},      # 自然界常见（牛背鹭），测试模型是否敢说Yes
    {"Sub": "plant", "Rel": "growing on", "Obj": "building", "Expected": "Yes"}, # 爬山虎、青苔等
    {"Sub": "sign", "Rel": "says", "Obj": "number", "Expected": "Yes"},          # 语言习惯在视觉中的体现（限速牌）
    {"Sub": "cat", "Rel": "looking at", "Obj": "book", "Expected": "Yes"},       # 猫不认字，但物理上盯着书看是完全合理的
    {"Sub": "dog", "Rel": "covered in", "Obj": "snow", "Expected": "Yes"},       # 视觉常识：狗在雪地里玩耍
    {"Sub": "tree", "Rel": "growing on", "Obj": "hill", "Expected": "Yes"},      # 标准自然场景
    {"Sub": "bag", "Rel": "hanging from", "Obj": "branch", "Expected": "Yes"},   # 包挂在树枝上，视觉合理
    {"Sub": "bird", "Rel": "flying in", "Obj": "room", "Expected": "Yes"},       # 相比船在房间飞(No)，鸟飞进房间是常见意外(Yes)
    {"Sub": "person", "Rel": "riding", "Obj": "surfboard", "Expected": "Yes"},   # 冲浪的正确描述
    {"Sub": "door", "Rel": "of", "Obj": "car", "Expected": "Yes"},               # 从属关系对照
    {"Sub": "snow", "Rel": "covering", "Obj": "car", "Expected": "Yes"},         # 典型的冬季视觉场景
    {"Sub": "boat", "Rel": "on", "Obj": "beach", "Expected": "Yes"},             # 搁浅或停靠的船
    {"Sub": "man", "Rel": "carrying", "Obj": "box", "Expected": "Yes"},          # 典型人类动作
    {"Sub": "clock", "Rel": "on", "Obj": "building", "Expected": "Yes"},         # 钟楼/建筑上的钟
    {"Sub": "umbrella", "Rel": "covering", "Obj": "woman", "Expected": "Yes"},   # 撑伞的物理状态
    {"Sub": "pizza", "Rel": "in", "Obj": "box", "Expected": "Yes"},              # 日常物品包含关系
    {"Sub": "plane", "Rel": "above", "Obj": "mountain", "Expected": "Yes"},      # 正常的宏观空间关系
    {"Sub": "chair", "Rel": "near", "Obj": "table", "Expected": "Yes"}           # 标准室内陈设
]

# 2. 匹配你原代码中的问题生成逻辑
def format_question(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    if rel != 'and':
        if rel == 'wears': rel_formatted = 'wearing'
        elif rel == 'says': rel_formatted = 'saying'
        elif rel == 'has': rel_formatted = 'have'
        
        rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        return f"Can {sub_f} {rel_f} {obj_f}?"
    else:
        return f"Can {sub_f} {rel} {obj_f} appear together?"

# 3. 定义测试函数
def run_ablation(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 构建所有问题的 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 批量推理 (关闭 tqdm 防止 Jupyter 输出过长)
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    
    # 结果解析与统计
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        # 简单判定：如果模型输出包含预期答案（忽略大小写）则判对
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"Finish!  {len(df_results['Correct'])} samples' accuracy: {accuracy:.1f}%\n")
    return df_results

# Experiment

In [5]:
temperature = 0.

In [6]:
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=150, 
)

# Exp 1: Original prompts (v4)

In [7]:
prompt_v1 = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
Interpret this strictly as a statement about physical and commonsense plausibility in the real world.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
Respond with a Yes or No if both commonsense and visually plausible, without additional explainations."""

shots_v1 = [
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can an animal be eating an apple?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can a bird be under a mountain?"},
    {"role": "assistant", "content": "No"},
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "No"}
]

print("=== Exp 1 Original Prompt ===")
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=150, 
)
df_res1 = run_ablation(prompt_v1, shots_v1)
display(df_res1) 

=== Exp 1 Original Prompt ===
Finish!  50 samples' accuracy: 82.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Yes,True
1,Can a person be wearing a hat?,Yes,Yes,True
2,Can a boat be flying in a room?,No,No,True
3,Can a cat be sitting on a table?,Yes,Yes,True
4,Can a pizza be eating a man?,No,No,True
5,Can a train be parked on a track?,Yes,Yes,True
6,Can a tree be riding a bike?,No,No,True
7,Can a dog be holding a branch?,Yes,Yes,True
8,Can a flower be growing on a rock?,Yes,Yes,True
9,Can a logo be of a banana?,Yes,Yes,True


# Exp 2: Chain-of-Thought (CoT)

In [8]:
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
End your response with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"}
]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=500, 
)

print("=== Exp 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, [])
display(df_res2)



=== Exp 2: Chain-of-Thought (CoT) ===
Finish!  50 samples' accuracy: 96.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"To evaluate this question, let's break it down step by step:\n\n1. A car is a common, everyday object that exists in the real world.\n2. Streets are also common, everyday features found in many urban and rural areas.\n3. Cars are designed to be mobile and can move on various surfaces, including roads and streets.\n4. Streets are typically paved or constructed to accommodate vehicles, providing a surface for cars to drive on.\n\nConsidering these points, it is physically and visually plausible for a car to be on a street. In fact, this is a very common scenario that occurs frequently in everyday life. It's also a situation that could easily be captured in a real photograph or video.\n\nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,"To evaluate this question, let's break it down step by step:\n\n1. A person is a living being with a physical body.\n2. A hat is a type of headgear designed to be worn on the head.\n3. People typically wear hats to protect themselves from the sun, cold, or as a fashion statement.\n4. Hats are designed to fit on a person's head, which is a common and practical use.\n\nConsidering these points, it is physically and visually plausible for a person to wear a hat. People wear hats in everyday life, and it's a common sight in real-world scenarios. Therefore, it's possible to capture a photograph or video of a person wearing a hat.\n\nFinal Answer: Yes",True
2,Can a boat be flying in a room?,No,"To evaluate this question, let's break it down step by step:\n\n1. **Physical Plausibility**: A boat is a large, heavy object designed for water travel. It is not designed to fly or operate in air. The physical properties of a boat, such as its weight, size, and aerodynamics, make it impossible for it to generate lift and sustain flight on its own.\n\n2. **Visual Plausibility**: Considering a real-world scenario where a boat is somehow magically or artificially made to fly in a room, it would be extremely difficult to achieve. The room would need to be enormous to accommodate the size of a boat, and the boat would need to be somehow suspended or propelled in mid-air without any visible support or mechanism. Additionally, the visual cues of a boat in a room would be highly unusual and unlikely to be seen in everyday life.\n\n3. **Real-World Conditions**: In typical real-world conditions, boats are designed for water, not air. They are not equipped with the necessary features to fly, and there is no known technology that can make a boat fly in a room without any external support or artificial means.\n\nConsidering these factors, the situation of a boat flying in a room is not physically or visually plausible in the real world. It is not something that could realistically appear in a real photograph or video without any CGI or fantasy elements.\n\nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,"To evaluate this question, let's break it down step by step:\n\n1. **Physical Plausibility:** Cats are living creatures that can move around and sit on various surfaces. Tables are common household items with flat surfaces. It's entirely possible for a cat to climb onto a table, either by jumping or being placed there by a person.\n\n2. **Visual Plausibility:** A cat sitting on a table is a common and realistic scenario. It's something that can easily be captured in a photograph or video, especially in a home setting where cats and tables are common.\n\nGiven these considerations, the scenario of a cat sitting on a table is both physically and visually plausible in the real world.\n\nFinal Answer: Yes",True
4,Can a pizza be eating a man?,No,"To evaluate this question, let's break it down step by step:\n\n1. A pizza is a type of food, typically made from dough, sauce, cheese, and various toppings. It's a physical object that exists in the real world.\n2. A man is a human being, a living organism with a physical body.\n3. The act

In [9]:
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: An arm is a dependent biological part, not an independent object.\n- [Geometry]: Sky implies unattached aerial position. An arm lacks aerodynamic structure.\n- [Agency]: Arms cannot fly. This violates basic physics.\nFinal Answer: No"},
    
    {"role": "user", "content": "Can a person be wearing a hat?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A hat is sized to fit a person's head.\n- [Geometry]: 'Wearing' implies placed on the head. This is a standard spatial configuration.\n- [Agency]: A person has the physical capacity to wear items.\nFinal Answer: Yes"},
    
    {"role": "user", "content": "Can a vase be sitting on an airplane?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A vase is small and fragile. An airplane is massive.\n- [Geometry]: 'Sitting on' an airplane strictly means the exterior roof/wings. A vase cannot physically balance on the exterior of a plane under typical conditions.\n- [Agency]: Objects do not naturally sit on moving vehicle exteriors.\nFinal Answer: No"}
]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=500, 
)

print("=== Exp 3: Chain-of-Thought (CoT) v2 ===")
df_res2 = run_ablation(prompt_cot, [])
display(df_res2)




=== Exp 3: Chain-of-Thought (CoT) v2 ===
Finish!  50 samples' accuracy: 92.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Reasoning:\n- [Scale/Part]: A car is a typical object of a size that can be placed on a street, which is a large, flat surface.\n- [Geometry]: A car can be aligned with a street, as it is designed to be driven on roads and streets.\n- [Agency]: A car is capable of being placed on a street, as it is a common and intended use of the vehicle.\nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,"Reasoning:\n- [Scale/Part]: A person is a large object, and a hat is a smaller object. This size difference is plausible.\n- [Geometry]: A person's head is a flat surface, and a hat is designed to fit on a head. The geometry of a person's head and a hat is compatible.\n- [Agency]: A person has the capability to wear a hat, as it is a common and everyday activity.\nFinal Answer: Yes",True
2,Can a boat be flying in a room?,No,"Reasoning:\n- [Scale/Part]: A boat is typically a large object, and a room is a confined space. The size of a boat usually exceeds the dimensions of a room, making it physically impossible for a full-sized boat to fit inside a room.\n- [Geometry]: Even if a boat were somehow miniaturized to fit inside a room, its shape and structure would not allow it to be suspended in mid-air within the room. Boats are designed for water, not for flight, and their design does not accommodate the necessary aerodynamics for flight.\n- [Agency]: There is no known technology or mechanism that would allow a boat to defy gravity and fly within a room.\nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,"Reasoning:\n- [Scale/Part]: A cat is a relatively small animal, and a table is a piece of furniture designed to hold objects. The size of a cat is compatible with the size of a table, so this is plausible.\n- [Geometry]: Cats can balance on their legs, and tables have a flat surface. It's physically possible for a cat to sit on a table, as long as the table is stable and the cat can climb up.\n- [Agency]: Cats have the ability to climb, balance, and sit, so they have the capability to sit on a table.\nFinal Answer: Yes",True
4,Can a pizza be eating a man?,No,"Reasoning:\n- [Scale/Part]: A pizza is typically much smaller than a man, so it's unlikely to be consuming a person. However, considering a giant pizza, it's still not plausible for a pizza to consume a man in the classical sense, as pizzas don't have a digestive system or the ability to consume solid objects.\n- [Geometry]: Even if we consider a giant pizza, the geometry of a pizza (a flat, round, and relatively thin object) doesn't allow it to physically engulf or consume a man, who is a three-dimensional object.\n- [Agency]: Pizzas don't have the capability to move or act on their own, let alone consume a person.\n\nFinal Answer: No",True
5,Can a train be parked on a track?,Yes,"Reasoning:\n- [Scale/Part]: A train is a large vehicle, and a track is a structure designed to support and guide it. The size of a train matches the scale of a track, so this is plausible.\n- [Geometry]: A train is designed to run on a track, and the track's geometry is designed to accommodate the train's wheels. The alignment of a train on a track is physically possible.\n- [Agency]: A train has the capability to be placed on a track, either by human operation or automated systems.\nFinal Answer: Yes",True
6,Can a tree be riding a bike?,No,"Reasoning:\n- [Scale/Part]: A tree is typically much larger and heavier than a bike, making it physically impossible for a tree to ride a bike. Trees are stationary objects and do not have the ability to move on their own.\n- [Geometry]: Even if we consider a tree somehow managing to sit on a bike, the tree's massive size and weight would make it impossible to balance on the bike, and the bike's frame would likely be crushed under the tree's weight.\n- [Agency]: Trees do not have the capability to operate or control a bike, as they are inanimate objects and do not possess the ne

In [10]:
from collections import Counter
import re
import pandas as pd

# 1. 重新配置 Self-consistency 的采样参数
# 【关键改动】
# temperature: 调高到 0.7 引入随机性，确保模型每次的思维路径不一样
# n=5: 让 vLLM 一次性并发生成 5 个不同的回答，极大地节省时间
sampling_params_sc = SamplingParams(temperature=0.01, top_p=0.95, n=5, max_tokens=256)

# 复用你设计得非常好的 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = []

# 2. 定义专属的 Self-Consistency 运行函数
def run_self_consistency(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 组装所有 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 并行生成（此时 outputs 中的每个 out 将包含 5 个生成结果）
    outputs = llm.generate(prompts, sampling_params_sc, use_tqdm=False)
    
    results = []
    for case, out in zip(test_cases, outputs):
        votes = []
        
        # 遍历同一个 prompt 生成的 5 个不同推理过程
        for o in out.outputs:
            text = o.text.strip()
            
            # 使用正则严格匹配你设定的 Final Answer: Yes/No 格式
            match = re.search(r"Final Answer:\s*(Yes|No)", text, re.IGNORECASE)
            if match:
                votes.append(match.group(1).capitalize()) # 统一转为 Yes 或 No
            else:
                # 兜底逻辑：如果模型没按格式输出，从最后 20 个字符找 yes/no
                last_chars = text.lower()[-20:]
                if 'yes' in last_chars: votes.append('Yes')
                elif 'no' in last_chars: votes.append('No')

        # 多数投票机制 (Majority Voting)
        if not votes:
            final_vote = "Error"
        else:
            vote_counts = Counter(votes)
            # 取出得票数最高的答案 (例如: [('No', 4), ('Yes', 1)])
            final_vote = vote_counts.most_common(1)[0][0] 
            
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Votes_Detail": str(dict(Counter(votes))), # 方便你观察票数分布 (例如 "{'No': 4, 'Yes': 1}")
            "Final_Vote": final_vote,
            "Correct": final_vote.lower() == case['Expected'].lower()
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> Self-Consistency 测试完成! 准确率为: {accuracy:.1f}%\n")
    return df_results

# 3. 运行测试
print("=== Exp 3: Self-Consistency (SC) ===")
pd.set_option('display.max_colwidth', None)
df_res_sc = run_self_consistency(prompt_cot, shots_cot)
display(df_res_sc)

=== Exp 3: Self-Consistency (SC) ===

=> Self-Consistency 测试完成! 准确率为: 76.0%



,Question,Expected,Votes_Detail,Final_Vote,Correct
0,Can a car be on a street?,Yes,{'Yes': 5},Yes,True
1,Can a person be wearing a hat?,Yes,{'Yes': 5},Yes,True
2,Can a boat be flying in a room?,No,{'No': 5},No,True
3,Can a cat be sitting on a table?,Yes,{'Yes': 5},Yes,True
4,Can a pizza be eating a man?,No,{'No': 5},No,True
5,Can a train be parked on a track?,Yes,{'Yes': 5},Yes,True
6,Can a tree be riding a bike?,No,{'No': 5},No,True
7,Can a dog be holding a branch?,Yes,{'Yes': 5},Yes,True
8,Can a flower be growing on a rock?,Yes,{'Yes': 5},Yes,True
9,Can a logo be of a banana?,Yes,{'Yes': 5},Yes,True


In [11]:
from collections import Counter
import re
import pandas as pd

# 1. 重新配置 Self-consistency 的采样参数
# 【关键改动】
# temperature: 调高到 0.7 引入随机性，确保模型每次的思维路径不一样
# n=5: 让 vLLM 一次性并发生成 5 个不同的回答，极大地节省时间
sampling_params_sc = SamplingParams(temperature=0.1, top_p=0.95, n=5, max_tokens=256)

# 复用你设计得非常好的 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: An arm is a dependent biological part, not an independent object.\n- [Geometry]: Sky implies unattached aerial position. An arm lacks aerodynamic structure.\n- [Agency]: Arms cannot fly. This violates basic physics.\nFinal Answer: No"},
    
    {"role": "user", "content": "Can a person be wearing a hat?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A hat is sized to fit a person's head.\n- [Geometry]: 'Wearing' implies placed on the head. This is a standard spatial configuration.\n- [Agency]: A person has the physical capacity to wear items.\nFinal Answer: Yes"},
    
    {"role": "user", "content": "Can a vase be sitting on an airplane?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A vase is small and fragile. An airplane is massive.\n- [Geometry]: 'Sitting on' an airplane strictly means the exterior roof/wings. A vase cannot physically balance on the exterior of a plane under typical conditions.\n- [Agency]: Objects do not naturally sit on moving vehicle exteriors.\nFinal Answer: No"}
]

# 2. 定义专属的 Self-Consistency 运行函数
def run_self_consistency(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 组装所有 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 并行生成（此时 outputs 中的每个 out 将包含 5 个生成结果）
    outputs = llm.generate(prompts, sampling_params_sc, use_tqdm=False)
    
    results = []
    for case, out in zip(test_cases, outputs):
        votes = []
        
        # 遍历同一个 prompt 生成的 5 个不同推理过程
        for o in out.outputs:
            text = o.text.strip()
            
            # 使用正则严格匹配你设定的 Final Answer: Yes/No 格式
            match = re.search(r"Final Answer:\s*(Yes|No)", text, re.IGNORECASE)
            if match:
                votes.append(match.group(1).capitalize()) # 统一转为 Yes 或 No
            else:
                # 兜底逻辑：如果模型没按格式输出，从最后 20 个字符找 yes/no
                last_chars = text.lower()[-20:]
                if 'yes' in last_chars: votes.append('Yes')
                elif 'no' in last_chars: votes.append('No')

        # 多数投票机制 (Majority Voting)
        if not votes:
            final_vote = "Error"
        else:
            vote_counts = Counter(votes)
            # 取出得票数最高的答案 (例如: [('No', 4), ('Yes', 1)])
            final_vote = vote_counts.most_common(1)[0][0] 
            
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Votes_Detail": str(dict(Counter(votes))), # 方便你观察票数分布 (例如 "{'No': 4, 'Yes': 1}")
            "Final_Vote": final_vote,
            "Correct": final_vote.lower() == case['Expected'].lower()
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> Self-Consistency 测试完成! 准确率为: {accuracy:.1f}%\n")
    return df_results

# 3. 运行测试
print("=== Exp 3: Self-Consistency (SC) ===")
pd.set_option('display.max_colwidth', None)
df_res_sc = run_self_consistency(prompt_cot, shots_cot)
display(df_res_sc)

=== Exp 3: Self-Consistency (SC) ===

=> Self-Consistency 测试完成! 准确率为: 84.0%



,Question,Expected,Votes_Detail,Final_Vote,Correct
0,Can a car be on a street?,Yes,{'Yes': 5},Yes,True
1,Can a person be wearing a hat?,Yes,{'Yes': 5},Yes,True
2,Can a boat be flying in a room?,No,{'No': 5},No,True
3,Can a cat be sitting on a table?,Yes,{'Yes': 5},Yes,True
4,Can a pizza be eating a man?,No,{'No': 5},No,True
5,Can a train be parked on a track?,Yes,{'Yes': 5},Yes,True
6,Can a tree be riding a bike?,No,{'No': 5},No,True
7,Can a dog be holding a branch?,Yes,{'Yes': 5},Yes,True
8,Can a flower be growing on a rock?,Yes,{'Yes': 5},Yes,True
9,Can a logo be of a banana?,Yes,{'Yes': 5},Yes,True


In [12]:
import pandas as pd
from collections import Counter
import re

# ==========================================
# 1. 锁死温度，确保每次推理绝对稳定
# ==========================================
sampling_params_ensemble = SamplingParams(temperature=0, max_tokens=256)
original_params = sampling_params
sampling_params = sampling_params_ensemble

# ==========================================
# 2. 定义三个不同视角的 Prompt 与对应的 Few-shot
# ==========================================

# 视角 A：严谨的物理与几何专家 (专注尺度、重力、空间相容性)
prompt_physics = """You are a strict physics engine evaluator. 
Determine if the relationship between the Subject and Object is physically possible in the real world.
Focus strictly on scale, gravity, structural integrity, and anatomical rules.
Ignore fictional, metaphorical, or magic scenarios.
Respond step-by-step focusing on physical geometry, ending with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_physics = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: An arm is a biological appendage. It lacks aerodynamic surfaces (wings) and a propulsion mechanism. Gravity would pull it to the ground. \nFinal Answer: No"}
]

# 视角 B：挑剔的纪实摄影师 (专注视觉可行性和日常出现概率)
prompt_photo = """You are a strict documentary photographer. 
Determine if you could realistically take a raw, unedited photo of this scene in the real world today.
Reject anything that requires CGI, staged extreme edge cases, or surreal art setups.
Respond step-by-step focusing on visual realism, ending with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_photo = [
    {"role": "user", "content": "Can a train be parked on a street?"},
    {"role": "assistant", "content": "Reasoning: Trains require heavy iron tracks. While streetcars (trams) exist on streets, a standard heavy train parked on a standard asphalt street is not a typical real-world scene a photographer would encounter without a catastrophic derailment. \nFinal Answer: No"}
]

# 视角 C：依靠直觉的常识裁判 (专注人类日常经验，拒绝杠精)
prompt_commonsense = """You are a commonsense judge.
Evaluate if the given situation aligns with typical everyday human experience.
Do not overthink or find contrived excuses to make impossible things possible. If it sounds absurd to a normal person, reject it.
Respond step-by-step focusing on daily logic, ending with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_commonsense = [
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: Roads are built specifically for cars to drive on. This is the most standard mode of human transportation. \nFinal Answer: Yes"}
]

# ==========================================
# 3. 运行三次独立测试
# ==========================================
print("=== 开始 Prompt Ensembling (视角集成) 推理 ===")
print("正在运行视角 A: 物理学家...")
df_phys = run_ablation(prompt_physics, shots_physics)

print("正在运行视角 B: 纪实摄影师...")
df_phot = run_ablation(prompt_photo, shots_photo)

print("正在运行视角 C: 常识裁判...")
df_comm = run_ablation(prompt_commonsense, shots_commonsense)

# ==========================================
# 4. 提取答案并进行多数投票 (Majority Voting)
# ==========================================
def extract_final_answer(text):
    match = re.search(r"Final Answer:\s*(Yes|No)", text, re.IGNORECASE)
    if match: return match.group(1).capitalize()
    
    last_chars = text.lower()[-20:]
    if 'yes' in last_chars: return 'Yes'
    if 'no' in last_chars: return 'No'
    return 'Error'

# 构建综合 DataFrame
df_ensemble = pd.DataFrame({
    'Question': df_phys['Question'],
    'Expected': df_phys['Expected'],
    'Ans_Physics': df_phys['Model_Output'].apply(extract_final_answer),
    'Ans_Photo': df_phot['Model_Output'].apply(extract_final_answer),
    'Ans_Common': df_comm['Model_Output'].apply(extract_final_answer)
})

# 投票逻辑
def get_majority_vote(row):
    votes = [row['Ans_Physics'], row['Ans_Photo'], row['Ans_Common']]
    valid_votes = [v for v in votes if v in ['Yes', 'No']]
    if not valid_votes: return "Error"
    return Counter(valid_votes).most_common(1)[0][0]

df_ensemble['Final_Vote'] = df_ensemble.apply(get_majority_vote, axis=1)
df_ensemble['Correct'] = df_ensemble['Final_Vote'] == df_ensemble['Expected']

# 还原参数
sampling_params = original_params

# ==========================================
# 5. 展示结果
# ==========================================
accuracy = df_ensemble['Correct'].mean() * 100
print(f"\n=> Prompt Ensembling 测试完成! 最终准确率为: {accuracy:.1f}%\n")

# 查看模型出现分歧（2:1）或者集体翻车的具体案例
display(df_ensemble[['Question', 'Expected', 'Ans_Physics', 'Ans_Photo', 'Ans_Common', 'Final_Vote', 'Correct']])

=== 开始 Prompt Ensembling (视角集成) 推理 ===
正在运行视角 A: 物理学家...
Finish!  50 samples' accuracy: 92.0%

正在运行视角 B: 纪实摄影师...
Finish!  50 samples' accuracy: 94.0%

正在运行视角 C: 常识裁判...
Finish!  50 samples' accuracy: 86.0%


=> Prompt Ensembling 测试完成! 最终准确率为: 90.0%



,Question,Expected,Ans_Physics,Ans_Photo,Ans_Common,Final_Vote,Correct
0,Can a car be on a street?,Yes,Yes,Yes,Yes,Yes,True
1,Can a person be wearing a hat?,Yes,Yes,Yes,Yes,Yes,True
2,Can a boat be flying in a room?,No,No,No,No,No,True
3,Can a cat be sitting on a table?,Yes,Yes,Yes,Yes,Yes,True
4,Can a pizza be eating a man?,No,No,No,No,No,True
5,Can a train be parked on a track?,Yes,Yes,Yes,Yes,Yes,True
6,Can a tree be riding a bike?,No,No,No,No,No,True
7,Can a dog be holding a branch?,Yes,Yes,Yes,Yes,Yes,True
8,Can a flower be growing on a rock?,Yes,No,Yes,No,No,False
9,Can a logo be of a banana?,Yes,Yes,Yes,Yes,Yes,True


ERROR 05-14 17:10:18 [core_client.py:667] Engine core proc EngineCore died unexpectedly, shutting down client.


/home/shsi/.conda/envs/promptllm/lib/python3.10/multiprocessing/resource_tracker.py:224: UserWarning: resource_tracker: There appear to be 1 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
